# Fixes for LTR Dataset Reformulation

**Goal:** Apply two fixes to the rank datasets produced by `05_LTR_Dataset_Reformulation.ipynb`:

1. **Critical:** Remove `YearMonth` string column that slipped into `FEATURE_COLS` — XGBoost cannot handle strings
2. **Recommended:** Re-add `Task_Skills` as multi-hot binary columns — skill features are NOT leakage (task text is available at inference time)

**Overwrites:** `rank_train.csv`, `rank_val.csv`, `rank_test.csv`, `dataset_metadata.json`

## Step A1: Load Existing Rank Datasets and Original CSVs

In [1]:
import pandas as pd
import numpy as np
import json
import os

rank_train = pd.read_csv("../data/processed/rank_datasets/rank_train.csv")
rank_val   = pd.read_csv("../data/processed/rank_datasets/rank_val.csv")
rank_test  = pd.read_csv("../data/processed/rank_datasets/rank_test.csv")

orig_train = pd.read_csv("../data/processed/split_datasets/feature_engineered_train_dataset.csv")
orig_val   = pd.read_csv("../data/processed/split_datasets/feature_engineered_val_dataset.csv")
orig_test  = pd.read_csv("../data/processed/split_datasets/feature_engineered_test_dataset.csv")

print(f"Rank train: {rank_train.shape}, val: {rank_val.shape}, test: {rank_test.shape}")
print(f"Original train: {orig_train.shape}, val: {orig_val.shape}, test: {orig_test.shape}")
print(f"\nRank train columns ({len(rank_train.columns)}):")
print(list(rank_train.columns))

Rank train: (36605, 45), val: (7829, 44), test: (7827, 44)
Original train: (8314, 38), val: (1588, 38), test: (1728, 38)

Rank train columns (45):
['Task_ID', 'Estimated_Planned_Hours', 'Planned_Hours_Log', 'Task_Text_Length', 'Task_Word_Count', 'Task_Description_Length', 'Task_Name_Length', 'Has_Task_Description', 'Has_Deadline', 'Days_To_Deadline', 'Task_Skill_Count', 'Project_Task_Count_Train', 'Project_Avg_Skill_Count_Train', 'Project_Avg_Planned_Hours_Train', 'Created_Year', 'Created_Month', 'Created_DayOfWeek', 'Created_Quarter', 'Task_Priority', 'Employee_Department', 'Employee_Job_Position', 'Task_Name', 'Task_Text', 'Date', 'Task_Description', 'Project_Name', 'Created_Date', 'Deadline_Date', 'Planned_Task_Size', 'YearMonth', 'Employee_ID', 'relevance', 'historical_task_count', 'log_historical_task_count', 'historical_avg_planned_hours', 'historical_avg_skill_count', 'historical_unique_projects', 'historical_pct_medium', 'historical_pct_large', 'sample_weight', 'Project_Name_en

## Step A2: Verify the YearMonth Bug

In [2]:
if "YearMonth" in rank_train.columns:
    print(f"YearMonth column dtype: {rank_train['YearMonth'].dtype}")
    print(f"Sample values: {rank_train['YearMonth'].head(3).tolist()}")
    print("This will cause XGBoost to error or silently degrade. Must remove.")
else:
    print("YearMonth not in columns (already fixed)")

string_cols_in_rank = [
    c for c in rank_train.columns
    if not pd.api.types.is_numeric_dtype(rank_train[c])
]
print(f"\nAll non-numeric columns currently in rank_train:")
for c in string_cols_in_rank:
    print(f"  - {c}: {rank_train[c].dtype} | sample: {rank_train[c].iloc[0]}")

YearMonth column dtype: str
Sample values: ['2025-04', '2025-04', '2025-04']
This will cause XGBoost to error or silently degrade. Must remove.

All non-numeric columns currently in rank_train:
  - Task_ID: str | sample: TSK-1
  - Task_Priority: str | sample: Low
  - Employee_Department: str | sample: Administration
  - Employee_Job_Position: str | sample: Software Engineer
  - Task_Name: str | sample: training
  - Task_Text: str | sample: training training. work details: analysis | inventory module 'how to work location transactions" to asha for richi develpoment | inventory traning for asha | pos mobile app training | handover the pre costing and module to maleesha and malshi | papparich training | visit the papa rich and help them to practice modified process
  - Date: str | sample: 2025-04-16
  - Task_Description: str | sample: training. work details: analysis | inventory module 'how to work location transactions" to asha for richi develpoment | inventory traning for asha | pos mob

## Step A3: Extract Task_Skills per Unique Task

In [3]:
task_skills_train = orig_train[["Task_ID", "Task_Skills"]].drop_duplicates(subset=["Task_ID"])
task_skills_val   = orig_val[["Task_ID", "Task_Skills"]].drop_duplicates(subset=["Task_ID"])
task_skills_test  = orig_test[["Task_ID", "Task_Skills"]].drop_duplicates(subset=["Task_ID"])

task_skills_all = (
    pd.concat([task_skills_train, task_skills_val, task_skills_test])
    .drop_duplicates(subset=["Task_ID"])
    .reset_index(drop=True)
)

print(f"Task_Skills coverage: {task_skills_all['Task_Skills'].notna().sum()} / {len(task_skills_all)} tasks")
print(f"Unique skill labels found:")
print(task_skills_all["Task_Skills"].value_counts().head(20))
print(f"\nSample rows:")
display(task_skills_all.head(5))

Task_Skills coverage: 2442 / 2442 tasks
Unique skill labels found:
Task_Skills
Uncategorized                                                1035
Documentation                                                 429
Software Testing                                               94
Odoo ERP Development                                           92
Client & Functional Support                                    81
Database Management                                            56
Project Management                                             53
Project Management | Documentation                             50
Documentation | Project Management                             47
Documentation | Odoo ERP Development                           39
Odoo ERP Development | Documentation                           32
Training & Mentorship                                          24
Client & Functional Support | Odoo ERP Development             22
Documentation | Database Management                            

,Task_ID,Task_Skills
0,TSK-1640,Odoo ERP Development
1,TSK-1881,Odoo ERP Development | Documentation
2,TSK-182,Odoo ERP Development
3,TSK-2884,Project Management | Documentation | Training ...
4,TSK-405,Documentation | Odoo ERP Development | Project...


## Step A4: Define Skill Taxonomy and Create Multi-Hot Features

In [4]:
SKILL_TAXONOMY = [
    "Odoo ERP Development",
    "Database Management",
    "Server Administration",
    "Project Management",
    "Software Testing",
    "Web Development",
    "Client & Functional Support",
    "Documentation",
    "Training & Mentorship",
]

def add_skill_features(rank_df, task_skills_df):
    out = rank_df.merge(task_skills_df[["Task_ID", "Task_Skills"]], on="Task_ID", how="left")
    out["Task_Skills"] = out["Task_Skills"].fillna("")
    for skill in SKILL_TAXONOMY:
        col_name = "skill_" + skill.replace(" ", "_").replace("&", "and")
        out[col_name] = out["Task_Skills"].str.contains(skill, regex=False, na=False).astype(int)
    out["skill_is_uncategorized"] = (out["Task_Skills"] == "Uncategorized").astype(int)
    out = out.drop(columns=["Task_Skills"])
    return out

rank_train = add_skill_features(rank_train, task_skills_all)
rank_val   = add_skill_features(rank_val,   task_skills_all)
rank_test  = add_skill_features(rank_test,  task_skills_all)

new_skill_cols = (
    ["skill_" + s.replace(" ", "_").replace("&", "and") for s in SKILL_TAXONOMY]
    + ["skill_is_uncategorized"]
)

print(f"Added {len(new_skill_cols)} skill feature columns:")
print(f"  {new_skill_cols}")
print(f"\nSkill feature sums in rank_train (non-zero = active):")
print(rank_train[new_skill_cols].sum().to_string())

Added 10 skill feature columns:
  ['skill_Odoo_ERP_Development', 'skill_Database_Management', 'skill_Server_Administration', 'skill_Project_Management', 'skill_Software_Testing', 'skill_Web_Development', 'skill_Client_and_Functional_Support', 'skill_Documentation', 'skill_Training_and_Mentorship', 'skill_is_uncategorized']

Skill feature sums in rank_train (non-zero = active):
skill_Odoo_ERP_Development              6429
skill_Database_Management               1749
skill_Server_Administration              318
skill_Project_Management                5239
skill_Software_Testing                  3650
skill_Web_Development                    168
skill_Client_and_Functional_Support     4177
skill_Documentation                    13110
skill_Training_and_Mentorship           1121
skill_is_uncategorized                 15377


## Step A5: Remove YearMonth and Other String Columns from Features

In [5]:
EXCLUDE_FROM_FEATURES = [
    "Task_ID", "Employee_ID", "relevance", "sample_weight",
    "Task_Name", "Task_Text", "Task_Description", "Project_Name",
    "Task_Priority", "Employee_Department", "Employee_Job_Position",
    "Planned_Task_Size", "Date", "Created_Date", "Deadline_Date",
    "YearMonth",
    "Task_Skills",
]

FEATURE_COLS = [c for c in rank_train.columns if c not in EXCLUDE_FROM_FEATURES]

print(f"Total features after fix: {len(FEATURE_COLS)}")
print(f"\nFull feature list:")
for i, c in enumerate(FEATURE_COLS, 1):
    print(f"  {i:02d}. {c}")

Total features after fix: 39

Full feature list:
  01. Estimated_Planned_Hours
  02. Planned_Hours_Log
  03. Task_Text_Length
  04. Task_Word_Count
  05. Task_Description_Length
  06. Task_Name_Length
  07. Has_Task_Description
  08. Has_Deadline
  09. Days_To_Deadline
  10. Task_Skill_Count
  11. Project_Task_Count_Train
  12. Project_Avg_Skill_Count_Train
  13. Project_Avg_Planned_Hours_Train
  14. Created_Year
  15. Created_Month
  16. Created_DayOfWeek
  17. Created_Quarter
  18. historical_task_count
  19. log_historical_task_count
  20. historical_avg_planned_hours
  21. historical_avg_skill_count
  22. historical_unique_projects
  23. historical_pct_medium
  24. historical_pct_large
  25. Project_Name_enc
  26. Task_Priority_enc
  27. Employee_Department_enc
  28. Employee_Job_Position_enc
  29. Planned_Task_Size_enc
  30. skill_Odoo_ERP_Development
  31. skill_Database_Management
  32. skill_Server_Administration
  33. skill_Project_Management
  34. skill_Software_Testing
  35.

## Step A6: Verify All Features Are Numeric

In [6]:
non_numeric = []
for c in FEATURE_COLS:
    if not pd.api.types.is_numeric_dtype(rank_train[c]):
        non_numeric.append((c, rank_train[c].dtype))

if non_numeric:
    print(f"WARNING: {len(non_numeric)} non-numeric features detected:")
    for col, dtype in non_numeric:
        print(f"  - {col}: {dtype}")
        rank_train[col] = pd.to_numeric(rank_train[col], errors="coerce")
        rank_val[col]   = pd.to_numeric(rank_val[col], errors="coerce")
        rank_test[col]  = pd.to_numeric(rank_test[col], errors="coerce")
        print(f"    -> Coerced to numeric")
else:
    print(f"All {len(FEATURE_COLS)} features are numeric")

print(f"\nFeature dtypes:")
print(rank_train[FEATURE_COLS].dtypes.value_counts().to_string())

All 39 features are numeric

Feature dtypes:
int64      24
float64    15


## Step A7: Re-Sort by Task_ID and Recompute Group Sizes

In [7]:
rank_train = rank_train.sort_values("Task_ID").reset_index(drop=True)
rank_val   = rank_val.sort_values("Task_ID").reset_index(drop=True)
rank_test  = rank_test.sort_values("Task_ID").reset_index(drop=True)

def save_group_sizes(rank_df, name):
    group_sizes = rank_df.groupby("Task_ID", sort=False).size().tolist()
    path = f"../data/processed/rank_datasets/group_sizes_{name}.json"
    with open(path, "w") as f:
        json.dump(group_sizes, f)
    print(f"{name}: {len(group_sizes)} groups, mean size {np.mean(group_sizes):.1f}, total rows {sum(group_sizes)}")
    return group_sizes

train_groups = save_group_sizes(rank_train, "train")
val_groups   = save_group_sizes(rank_val,   "val")
test_groups  = save_group_sizes(rank_test,  "test")

train: 1709 groups, mean size 21.4, total rows 36605
val: 366 groups, mean size 21.4, total rows 7829
test: 367 groups, mean size 21.3, total rows 7827


## Step A8: Save Updated Datasets

In [8]:
output_dir = "../data/processed/rank_datasets"
os.makedirs(output_dir, exist_ok=True)

rank_train.to_csv(f"{output_dir}/rank_train.csv", index=False)
rank_val.to_csv(f"{output_dir}/rank_val.csv",     index=False)
rank_test.to_csv(f"{output_dir}/rank_test.csv",   index=False)

print("Updated rank datasets saved.")
print(f"Train: {rank_train.shape}")
print(f"Val:   {rank_val.shape}")
print(f"Test:  {rank_test.shape}")
print(f"\nColumns ({len(rank_train.columns)}):")
print(list(rank_train.columns))

Updated rank datasets saved.
Train: (36605, 55)
Val:   (7829, 54)
Test:  (7827, 54)

Columns (55):
['Task_ID', 'Estimated_Planned_Hours', 'Planned_Hours_Log', 'Task_Text_Length', 'Task_Word_Count', 'Task_Description_Length', 'Task_Name_Length', 'Has_Task_Description', 'Has_Deadline', 'Days_To_Deadline', 'Task_Skill_Count', 'Project_Task_Count_Train', 'Project_Avg_Skill_Count_Train', 'Project_Avg_Planned_Hours_Train', 'Created_Year', 'Created_Month', 'Created_DayOfWeek', 'Created_Quarter', 'Task_Priority', 'Employee_Department', 'Employee_Job_Position', 'Task_Name', 'Task_Text', 'Date', 'Task_Description', 'Project_Name', 'Created_Date', 'Deadline_Date', 'Planned_Task_Size', 'YearMonth', 'Employee_ID', 'relevance', 'historical_task_count', 'log_historical_task_count', 'historical_avg_planned_hours', 'historical_avg_skill_count', 'historical_unique_projects', 'historical_pct_medium', 'historical_pct_large', 'sample_weight', 'Project_Name_enc', 'Task_Priority_enc', 'Employee_Department_en

## Step A9: Update Metadata

In [9]:
employee_profile = pd.read_csv(f"{output_dir}/employee_profile_train.csv")

EMPLOYEE_FEATURE_COLS = [
    "historical_task_count",
    "log_historical_task_count",
    "historical_avg_planned_hours",
    "historical_avg_skill_count",
    "historical_unique_projects",
    "historical_pct_medium",
    "historical_pct_large",
]

metadata = {
    "n_train_tasks": int(rank_train["Task_ID"].nunique()),
    "n_val_tasks": int(rank_val["Task_ID"].nunique()),
    "n_test_tasks": int(rank_test["Task_ID"].nunique()),
    "n_employees": 49,
    "negatives_per_task": 20,
    "feature_cols": FEATURE_COLS,
    "n_features": len(FEATURE_COLS),
    "skill_feature_cols": new_skill_cols,
    "employee_feature_cols": EMPLOYEE_FEATURE_COLS,
    "max_min_imbalance_ratio": float(
        employee_profile["historical_task_count"].max() /
        employee_profile["historical_task_count"].min()
    ),
    "leakage_columns_dropped": [
        "Hours_Spent",
        "FLAG_LEAKAGE_Actual_Hours_Spent",
        "FLAG_LEAKAGE_Timesheet_Work_Logs",
        "FLAG_LEAKAGE_Timesheet_Logs_Count",
        "FLAG_LEAKAGE_Task_Stage",
        "FLAG_LEAKAGE_All_Collaborating_Employees",
        "Debug_Skill_Scores",
    ],
    "fixes_applied": [
        "Removed YearMonth string column from features",
        "Re-added Task_Skills as multi-hot binary features (not leakage)",
    ],
    "formulation": "learning_to_rank",
    "imbalance_handling": "rank_objective + inverse_freq_weights + smart_negative_sampling",
}

with open(f"{output_dir}/dataset_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata updated.")
print(f"Total features: {len(FEATURE_COLS)}")
print(f"  Skill features:              {len(new_skill_cols)}")
print(f"  Employee history features:   {len(EMPLOYEE_FEATURE_COLS)}")
print(f"  Task/project/temporal/other: {len(FEATURE_COLS) - len(new_skill_cols) - len(EMPLOYEE_FEATURE_COLS)}")

Metadata updated.
Total features: 39
  Skill features:              10
  Employee history features:   7
  Task/project/temporal/other: 22


## Step A10: Final Validation

In [10]:
assert set(rank_train["Task_ID"]).isdisjoint(set(rank_val["Task_ID"]))
assert set(rank_train["Task_ID"]).isdisjoint(set(rank_test["Task_ID"]))
print("No Task_ID leakage")

for name, df in [("train", rank_train), ("val", rank_val), ("test", rank_test)]:
    pos_per_task = df[df["relevance"] == 1].groupby("Task_ID").size()
    assert (pos_per_task >= 1).all(), f"{name} has tasks with no positives"
print("Every task has at least one positive")

for c in FEATURE_COLS:
    assert pd.api.types.is_numeric_dtype(rank_train[c]), f"{c} is not numeric in train"
    assert pd.api.types.is_numeric_dtype(rank_val[c]),   f"{c} is not numeric in val"
    assert pd.api.types.is_numeric_dtype(rank_test[c]),  f"{c} is not numeric in test"
print(f"All {len(FEATURE_COLS)} features are numeric")

assert "YearMonth" not in FEATURE_COLS, "YearMonth still in features!"
print("YearMonth removed from features")

assert all(col in FEATURE_COLS for col in new_skill_cols), "Skill features missing!"
print(f"All {len(new_skill_cols)} skill features present")

for name, df, groups in [
    ("train", rank_train, train_groups),
    ("val",   rank_val,   val_groups),
    ("test",  rank_test,  test_groups)
]:
    assert sum(groups) == len(df), f"{name} group sizes do not match row count"
print("Group sizes match row counts")

for name, df in [("train", rank_train), ("val", rank_val), ("test", rank_test)]:
    pos_rate = df["relevance"].mean()
    print(f"  {name} positive rate: {pos_rate:.3f}")

print("\nAll validation checks passed")
print(f"\nFinal dataset ready for model training:")
print(f"  Train: {rank_train.shape[0]:,} (task, employee) pairs from {rank_train['Task_ID'].nunique()} tasks")
print(f"  Val:   {rank_val.shape[0]:,} pairs from {rank_val['Task_ID'].nunique()} tasks")
print(f"  Test:  {rank_test.shape[0]:,} pairs from {rank_test['Task_ID'].nunique()} tasks")
print(f"  Features: {len(FEATURE_COLS)} (all numeric)")

No Task_ID leakage
Every task has at least one positive
All 39 features are numeric
YearMonth removed from features
All 10 skill features present
Group sizes match row counts
  train positive rate: 0.066
  val positive rate: 0.065
  test positive rate: 0.062

All validation checks passed

Final dataset ready for model training:
  Train: 36,605 (task, employee) pairs from 1709 tasks
  Val:   7,829 pairs from 366 tasks
  Test:  7,827 pairs from 367 tasks
  Features: 39 (all numeric)
